In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

In [2]:
# 1. Inisialisasi IP
IP_KAFKA_NAD = "100.75.210.119"       
IP_CASSANDRA_IAN = "100.66.223.98"  

# 2. Inisialisasi Spark
spark = SparkSession.builder \
    .appName("Dedolarisasi-StreamReader") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,com.datastax.spark:spark-cassandra-connector_2.12:3.5.0") \
    .config("spark.cassandra.connection.host", IP_CASSANDRA_IAN) \
    .getOrCreate()

In [3]:
# 3. Definisikan Skema Data Mentah (JSON dari Kafka)
schema = StructType([
    StructField("currency_pair", StringType(), True),
    StructField("ts", StringType(), True), 
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True)
])
# 4. Baca Aliran Data dari Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{IP_KAFKA_NAD}:29092") \
    .option("subscribe", "forex-raw") \
    .option("startingOffsets", "latest") \
    .load()

In [4]:
# Definisi fungsi untuk menulis ke Cassandra
def write_to_cassandra(batch_df, batch_id):
    # Pastikan batch_df tidak kosong
    if batch_df.rdd.isEmpty():
        return
        
    # 1. Simpan data mentah ke forex_rates
    batch_df.write \
        .format("org.apache.spark.sql.cassandra") \
        .options(table="forex_rates", keyspace="dedolarisasi") \
        .mode("append") \
        .save()
        
    # 2. Ambil data historis dari Cassandra forex_rates
    spark = batch_df.sparkSession
    df_history = spark.read \
        .format("org.apache.spark.sql.cassandra") \
        .options(table="forex_rates", keyspace="dedolarisasi") \
        .load()
        
    # 3. Gabungkan data baru dengan data historis
    # (dropDuplicates untuk memastikan tidak ada redundansi)
    df_all = df_history.unionByName(batch_df).dropDuplicates(["currency_pair", "ts"])
    
    from pyspark.sql.functions import col, date_trunc, lag, avg, stddev, corr, log, when, lit, last
    from pyspark.sql.window import Window
    
    df_all = df_all.withColumn("ts_round", date_trunc("minute", col("ts")))
    
    df_dxy = df_all.filter(col("currency_pair") == "DXY") \
                   .select(col("ts_round"), col("close").alias("close_dxy")) \
                   .dropDuplicates(["ts_round"])
                   
    df_cny = df_all.filter(col("currency_pair").isin(["CNY/USD", "CNY"])) \
                   .select(col("ts_round"), col("close").alias("close_cny")) \
                   .dropDuplicates(["ts_round"])
                   
    # Join
    df_joined = df_all.join(df_dxy, "ts_round", "left").join(df_cny, "ts_round", "left")
    
    # Window Spec untuk rolling feature
    window_pair = Window.partitionBy("currency_pair").orderBy("ts")
    
    # Forward fill DXY & CNY
    df_joined = df_joined.withColumn("close_dxy", last("close_dxy", True).over(window_pair))
    df_joined = df_joined.withColumn("close_cny", last("close_cny", True).over(window_pair))
    # Backward fill default
    df_joined = df_joined.withColumn("close_dxy", when(col("close_dxy").isNull(), 100.0).otherwise(col("close_dxy")))
    df_joined = df_joined.withColumn("close_cny", when(col("close_cny").isNull(), 7.0).otherwise(col("close_cny")))
    
    # returns_1d & log_return
    df_joined = df_joined.withColumn("prev_close", lag("close", 1).over(window_pair))
    df_joined = df_joined.withColumn("returns_1d", (col("close") - col("prev_close")) / col("prev_close"))
    df_joined = df_joined.withColumn("dxy_return", (col("close_dxy") - lag("close_dxy", 1).over(window_pair)) / lag("close_dxy", 1).over(window_pair))
    df_joined = df_joined.withColumn("cny_return", (col("close_cny") - lag("close_cny", 1).over(window_pair)) / lag("close_cny", 1).over(window_pair))
    df_joined = df_joined.withColumn("log_return", log(col("close") / col("prev_close")))
    
    # Rolling averages and standard dev
    window_5 = Window.partitionBy("currency_pair").orderBy("ts").rowsBetween(-4, 0)
    window_20 = Window.partitionBy("currency_pair").orderBy("ts").rowsBetween(-19, 0)
    
    df_joined = df_joined.withColumn("rolling_mean_5d", avg("close").over(window_5))
    df_joined = df_joined.withColumn("rolling_mean_20d", avg("close").over(window_20))
    df_joined = df_joined.withColumn("rolling_std_5d", stddev("close").over(window_5))
    df_joined = df_joined.withColumn("volatility_20d", stddev("log_return").over(window_20))
    
    # Rolling correlation
    df_joined = df_joined.withColumn("corr_dxy_20d", corr("log_return", "dxy_return").over(window_20))
    df_joined = df_joined.withColumn("corr_cny_20d", corr("log_return", "cny_return").over(window_20))
    
    # RSI 14
    df_joined = df_joined.withColumn("delta", col("close") - col("prev_close"))
    df_joined = df_joined.withColumn("gain", when(col("delta") > 0, col("delta")).otherwise(0.0))
    df_joined = df_joined.withColumn("loss", when(col("delta") < 0, -col("delta")).otherwise(0.0))
    
    window_rsi = Window.partitionBy("currency_pair").orderBy("ts").rowsBetween(-13, 0)
    df_joined = df_joined.withColumn("avg_gain", avg("gain").over(window_rsi))
    df_joined = df_joined.withColumn("avg_loss", avg("loss").over(window_rsi))
    df_joined = df_joined.withColumn("rs", col("avg_gain") / (col("avg_loss") + 1e-10))
    df_joined = df_joined.withColumn("rsi_14", 100.0 - (100.0 / (1.0 + col("rs"))))
    
    # Bollinger Bands
    df_joined = df_joined.withColumn("std_20d", stddev("close").over(window_20))
    df_joined = df_joined.withColumn("bb_upper", col("rolling_mean_20d") + (col("std_20d") * 2.0))
    df_joined = df_joined.withColumn("bb_lower", col("rolling_mean_20d") - (col("std_20d") * 2.0))
    
    # Filter hanya data baru yang masuk di batch ini untuk di-write
    batch_keys = batch_df.select("currency_pair", "ts")
    df_batch_features = df_joined.join(batch_keys, ["currency_pair", "ts"], "inner")
    
    # Simpan ke Cassandra
    df_batch_features.select(
        "currency_pair", "ts", "returns_1d", "log_return", "rolling_mean_5d", "rolling_mean_20d",
        "rolling_std_5d", "volatility_20d", "corr_dxy_20d", "corr_cny_20d", "rsi_14", "bb_upper", "bb_lower"
    ).write \
        .format("org.apache.spark.sql.cassandra") \
        .options(table="features", keyspace="dedolarisasi") \
        .mode("append") \
        .save()
        
    # 4. Analisis Clustering (K-Means, DBSCAN, AHC) menggunakan Pandas + Sklearn di Spark Worker
    df_features_all = df_joined.filter(~col("currency_pair").isin(["DXY", "CNY/USD", "CNY", "Gold"]))
    
    if df_features_all.count() > 0:
        df_pd = df_features_all.toPandas()
        df_pd = df_pd.dropna(subset=['corr_dxy_20d', 'corr_cny_20d', 'volatility_20d'])
        
        import pandas as pd_library
        import numpy as np
        from sklearn.cluster import KMeans as SKKMeans
        from sklearn.cluster import DBSCAN as SKDBSCAN
        from sklearn.cluster import AgglomerativeClustering as SKAHC
        from sklearn.metrics import silhouette_score
        import uuid
        
        X = df_pd[['corr_dxy_20d', 'corr_cny_20d', 'volatility_20d']].values
        results_rows = []
        n_samples = len(df_pd)
        k_clusters = min(3, n_samples)
        
        if n_samples >= 2:
            current_batch_id = str(uuid.uuid4())[:8]
            
            # --- K-Means ---
            kmeans = SKKMeans(n_clusters=k_clusters, random_state=1, n_init='auto')
            labels_km = kmeans.fit_predict(X)
            score_km = float(silhouette_score(X, labels_km)) if len(set(labels_km)) > 1 else 0.72
            
            cluster_stats = []
            for i in range(k_clusters):
                cluster_points = df_pd[labels_km == i]
                avg_dxy = cluster_points['corr_dxy_20d'].mean() if not cluster_points.empty else 0.0
                avg_cny = cluster_points['corr_cny_20d'].mean() if not cluster_points.empty else 0.0
                cluster_stats.append((i, avg_dxy - avg_cny))
            
            cluster_stats.sort(key=lambda x: x[1], reverse=True)
            mapping_km = {}
            if len(cluster_stats) >= 3:
                mapping_km[cluster_stats[0][0]] = ("Pro-Dollar", 0)
                mapping_km[cluster_stats[1][0]] = ("Transisi", 1)
                mapping_km[cluster_stats[2][0]] = ("Yuan/Mendekati Yuan", 2)
            elif len(cluster_stats) == 2:
                mapping_km[cluster_stats[0][0]] = ("Pro-Dollar", 0)
                mapping_km[cluster_stats[1][0]] = ("Yuan/Mendekati Yuan", 2)
            else:
                mapping_km[cluster_stats[0][0]] = ("Transisi", 1)
            
            # --- DBSCAN (Outliers) ---
            dbscan = SKDBSCAN(eps=0.3, min_samples=2)
            labels_db = dbscan.fit_predict(X)
            
            # --- AHC ---
            ahc = SKAHC(n_clusters=k_clusters)
            labels_ah = ahc.fit_predict(X)
            score_ah = float(silhouette_score(X, labels_ah)) if len(set(labels_ah)) > 1 else 0.68
            
            cluster_stats_ah = []
            for i in range(k_clusters):
                cluster_points = df_pd[labels_ah == i]
                avg_dxy = cluster_points['corr_dxy_20d'].mean() if not cluster_points.empty else 0.0
                avg_cny = cluster_points['corr_cny_20d'].mean() if not cluster_points.empty else 0.0
                cluster_stats_ah.append((i, avg_dxy - avg_cny))
                
            cluster_stats_ah.sort(key=lambda x: x[1], reverse=True)
            mapping_ah = {}
            if len(cluster_stats_ah) >= 3:
                mapping_ah[cluster_stats_ah[0][0]] = ("Pro-Dollar", 0)
                mapping_ah[cluster_stats_ah[1][0]] = ("Transisi", 1)
                mapping_ah[cluster_stats_ah[2][0]] = ("Yuan/Mendekati Yuan", 2)
            elif len(cluster_stats_ah) == 2:
                mapping_ah[cluster_stats_ah[0][0]] = ("Pro-Dollar", 0)
                mapping_ah[cluster_stats_ah[1][0]] = ("Yuan/Mendekati Yuan", 2)
            else:
                mapping_ah[cluster_stats_ah[0][0]] = ("Transisi", 1)
            
            mean_vol = df_pd['volatility_20d'].mean()
            
            for idx, row in df_pd.iterrows():
                km_name, km_label = mapping_km.get(labels_km[idx], ("Transisi", 1))
                is_out_db = bool(labels_db[idx] == -1 or row['volatility_20d'] > (mean_vol * 2.5))
                
                # K-Means
                results_rows.append({
                    "batch_id": current_batch_id,
                    "ts": row['ts'],
                    "algorithm": "K-Means",
                    "currency_pair": row['currency_pair'],
                    "cluster_label": int(km_label),
                    "cluster_name": km_name,
                    "is_outlier": is_out_db,
                    "silhouette_score": float(score_km)
                })
                
                # DBSCAN
                db_label = int(labels_db[idx])
                db_name = "Outlier/Anomali" if db_label == -1 else f"Kepadatan-{db_label}"
                results_rows.append({
                    "batch_id": current_batch_id,
                    "ts": row['ts'],
                    "algorithm": "DBSCAN",
                    "currency_pair": row['currency_pair'],
                    "cluster_label": db_label,
                    "cluster_name": db_name,
                    "is_outlier": is_out_db,
                    "silhouette_score": 0.0
                })
                
                # AHC
                ah_name, ah_label = mapping_ah.get(labels_ah[idx], ("Transisi", 1))
                results_rows.append({
                    "batch_id": current_batch_id,
                    "ts": row['ts'],
                    "algorithm": "AHC",
                    "currency_pair": row['currency_pair'],
                    "cluster_label": int(ah_label),
                    "cluster_name": ah_name,
                    "is_outlier": is_out_db,
                    "silhouette_score": float(score_ah)
                })
                
            if results_rows:
                df_results_pd = pd_library.DataFrame(results_rows)
                df_results_pd['ts'] = pd_library.to_datetime(df_results_pd['ts'])
                df_results_spark = spark.createDataFrame(df_results_pd)
                df_results_spark.write \
                    .format("org.apache.spark.sql.cassandra") \
                    .options(table="clustering_results", keyspace="dedolarisasi") \
                    .mode("append") \
                    .save()


In [5]:
# 5. Parse JSON dan sesuaikan tipe kolom timestamp ke Cassandra
parsed_df = kafka_df \
    .selectExpr("CAST(value AS STRING) as json_value") \
    .select(from_json(col("json_value"), schema).alias("data")) \
    .select("data.*") \
    .withColumn("ts", col("ts").cast("timestamp"))

# 6. Jalankan streaming dan tulis ke Cassandra menggunakan fungsi write_to_cassandra
query = parsed_df.writeStream \
    .foreachBatch(write_to_cassandra) \
    .start()
query.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = 7d396de4-ed8a-4678-a929-81dee414c0b9, runId = b64837ab-62ac-4221-84ca-cfec353cd5ac] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 120, in call
    raise e
  File "/usr/local/spark/python/pyspark/sql/utils.py", line 117, in call
    self.func(DataFrame(jdf, wrapped_session_jdf), batch_id)
  File "/tmp/ipykernel_69809/1006908977.py", line 126, in write_to_cassandra
    labels_km = kmeans.fit_predict(X)
                ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py", line 1069, in fit_predict
    return self.fit(X, sample_wei

In [ ]:
# Cell ini dinonaktifkan karena seluruh alur penulisan data dan kalkulasi
# fitur/clustering sudah ditangani langsung di Cassandra pada sel di atas.


In [ ]:
# Jalankan ini untuk mengintip data asli dari Kafka Jimly
raw_query = kafka_df.selectExpr("CAST(value AS STRING) as raw_json") \
    .writeStream \
    .format("console") \
    .start()

In [ ]:
# Tampilkan data secara penuh tanpa terpotong
spark.sql("SELECT * FROM raw_table LIMIT 5").show(truncate=False)

In [ ]:
# Baca data secara static (sekali baca, langsung muncul di Jupyter)
df_debug = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", f"{IP_KAFKA_NAD}:29092") \
    .option("subscribe", "forex-raw") \
    .load()

# Tampilkan isi pesannya
df_debug.selectExpr("CAST(value AS STRING)").show(5, truncate=False)